# Renovaí — Orquestrador Unificado do Banco de Dados
**Disciplina:** Modelagem de Dados — 2º Ano (Instituto J&F Germinatech)

Este notebook automatiza o fluxo completo do ecossistema de dados relacional:
1. **Estruturação DDL:** Executa todos os 61 scripts SQL modularizados via `banco_relacional_sql/init.sql` em transação segura.
2. **Carga Massiva (Povoamento):** Executa a rotina de carga de volume com dados sintéticos verossímeis via `carga_volume_renovai.ipynb`.
3. **Catalogação de Metadados:** Executa `documentacao_demandas/catalogo_dados/catalogacao_dbRenovai.ipynb` persistindo metadados na tabela `public.catalogo_metadados_tabelas`.
4. **Validação Pós-Carga e Governança:** Executa contagens de registros (`SELECT COUNT`) e checagens de integridade em todas as tabelas operacionais e de governança.


## 1. Configuração e Conexão com o Banco de Dados

In [ ]:
import os
import sys
from pathlib import Path
import psycopg2
import psycopg2.extras
from dotenv import load_dotenv

# Localiza a raiz do repositório modelagem-de-dados
CURRENT_DIR = Path.cwd()
REPO_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "orquestracao" else CURRENT_DIR

# Carrega variáveis de ambiente
for env_path in [REPO_ROOT / ".env", CURRENT_DIR / ".env", REPO_ROOT.parent / ".env"]:
    if env_path.is_file():
        load_dotenv(dotenv_path=env_path)
        break
else:
    load_dotenv()

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "postgres")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASS = os.getenv("DB_PASS") or os.getenv("DB_PASSWORD", "postgres")
DB_SSLMODE = os.getenv("DB_SSLMODE", "prefer")

print(f"Alvo: {DB_HOST}:{DB_PORT}/{DB_NAME} (Usuário: {DB_USER})")
conn = psycopg2.connect(
    host=DB_HOST,
    port=int(DB_PORT),
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASS,
    sslmode=DB_SSLMODE,
)
psycopg2.extras.register_uuid()
print("Conexão com PostgreSQL estabelecida com sucesso!")

## 2. Execução da Estruturação DDL (banco_relacional_sql/init.sql)

In [ ]:
init_sql_path = REPO_ROOT / "banco_relacional_sql" / "init.sql"
base_dir = init_sql_path.parent
sql_files = []

with open(init_sql_path, "r", encoding="utf-8") as f:
    for line in f:
        clean = line.strip()
        if clean.startswith(r"\i"):
            rel_path = clean.split(maxsplit=1)[1].strip().strip("'\"")
            target_file = (base_dir / rel_path).resolve()
            if not target_file.is_file():
                raise FileNotFoundError(f"Arquivo ausente: {target_file}")
            sql_files.append((rel_path, target_file))

print(f"{len(sql_files)} scripts SQL mapeados a partir de {init_sql_path.name}.")
conn.autocommit = False
cur = conn.cursor()

try:
    for idx, (rel, path) in enumerate(sql_files, start=1):
        with open(path, "r", encoding="utf-8") as sf:
            content = sf.read()
        if content.strip():
            cur.execute(content)
        print(f"[{idx:02d}/{len(sql_files):02d}] OK - {rel}")
    conn.commit()
    print("\n-> Schema, triggers, procedures, views e dados iniciais comitados com sucesso!")
except Exception as e:
    conn.rollback()
    print(f"\n[ERRO] Falha no schema. Rollback realizado: {e}")
    raise

## 3. Execução da Carga de Volume (carga_inicial_povoamento)

In [ ]:
import json

notebook_path = REPO_ROOT / "carga_volume_renovai.ipynb"
conn.autocommit = True
cur = conn.cursor()

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

scope = {"conn": conn, "cur": cur, "__name__": "__main__"}
code_cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]

print(f"Executando {len(code_cells)} blocos de povoamento...")
for idx, cell in enumerate(code_cells, start=1):
    src = "".join(cell.get("source", []))
    if not src.strip():
        continue
    lines = [l for l in src.splitlines() if not l.strip().startswith(("%", "!"))]
    cleaned = "\n".join(lines)
    if "psycopg2.connect" in cleaned:
        scope["conn"] = conn
        scope["cur"] = cur
        continue
    if "cur.close()" in cleaned and "conn.close()" in cleaned:
        continue
    exec(cleaned, scope)
    first_line = src.strip().splitlines()[0][:45]
    print(f"[{idx:02d}/{len(code_cells):02d}] Concluído: {first_line}")

print("\n-> Povoamento de volume finalizado com sucesso!")

## 4. Execução da Catalogação Técnica de Metadados (catalogacao_dbRenovai.ipynb)

In [ ]:
catalog_path = REPO_ROOT / "documentacao_demandas" / "catalogo_dados" / "catalogacao_dbRenovai.ipynb"
conn.autocommit = True
cur = conn.cursor()

with open(catalog_path, "r", encoding="utf-8") as f:
    nb_cat = json.load(f)

scope_cat = {"conn": conn, "cur": cur, "__name__": "__main__"}
code_cells_cat = [c for c in nb_cat.get("cells", []) if c.get("cell_type") == "code"]

print(f"Executando {len(code_cells_cat)} blocos de catalogação técnica...")
for idx, cell in enumerate(code_cells_cat, start=1):
    src = "".join(cell.get("source", []))
    if not src.strip():
        continue
    lines = [l for l in src.splitlines() if not l.strip().startswith(("%", "!"))]
    cleaned = "\n".join(lines)
    exec(cleaned, scope_cat)
    first_line = src.strip().splitlines()[0][:45]
    print(f"[{idx:02d}/{len(code_cells_cat):02d}] Concluído: {first_line}")

cur.execute("SELECT COUNT(*) FROM public.catalogo_metadados_tabelas")
print(f"\n-> Catalogação persistida com sucesso! {cur.fetchone()[0]} tabelas registradas em public.catalogo_metadados_tabelas.")

## 5. Validação Pós-Carga e Governança (SELECT COUNT)

In [ ]:
tabelas = [
    "cargos", "status", "tipos_rateios", "enderecos", "categorias_materiais",
    "materiais", "cooperativas", "empresas", "perfis", "usuarios",
    "funcionarios", "telefones", "equipes", "equipes_cooperados", "pedidos",
    "pedido_itens", "pedidos_cooperativas", "negociacoes", "avaliacoes",
    "coletas", "triagens", "estoques", "movimentacoes_estoques", "despesas",
    "lancamentos_despesas", "rotas", "rotas_enderecos", "rateios",
    "rateios_funcionarios", "empresa_materiais_interesses", "empresa_cooperativas_favoritas",
    "catalogo_metadados_tabelas"
]

total = 0
print(f"{'Tabela':<35} | {'Registros':>10}")
print("-" * 48)
for t in tabelas:
    try:
        cur.execute(f"SELECT COUNT(*) FROM {t}")
        c = cur.fetchone()[0]
        total += c
        print(f"{t:<35} | {c:>10d}")
    except Exception as err:
        print(f"{t:<35} | [ERRO: {err}]")
print("-" * 48)
print(f"{'TOTAL GERAL DE LINHAS':<35} | {total:>10d}")

cur.execute("SELECT COUNT(*) FROM estoques WHERE quantidade_kg < 0")
print(f"\nEstoques negativos (< 0 kg): {cur.fetchone()[0]} (esperado: 0)")
cur.execute("SELECT COUNT(*) FROM avaliacoes WHERE avaliador_id = avaliado_id")
print(f"Avaliações com avaliador = avaliado: {cur.fetchone()[0]} (esperado: 0)")
cur.execute("SELECT COUNT(*) FROM public.catalogo_metadados_tabelas")
print(f"Metadados de entidades catalogadas: {cur.fetchone()[0]}")

## 6. Encerramento Seguro da Conexão

In [ ]:
if cur and not cur.closed:
    cur.close()
if conn and not conn.closed:
    conn.close()
print("Conexão com PostgreSQL encerrada com segurança.")